# Mechanism transfer on insects — the clean stand

The M2OR holdout is confounded two ways: (a) **measurement structure** — M2OR is sparse and
which receptor x molecule pairs were assayed clusters by study, so a receptor's
"responds-to-class-C" label lives on a biased subset the model can exploit non-mechanistically;
(b) **general tuning** — a broadly-tuned receptor binds a bit of everything, so "responds to
C" is mostly tuning breadth, which the refined embedding already encodes.

The insect matrices (Carey mosquito 50x110, Hallem-Carlson fly 24x110) fix (a) **by
construction**: every receptor is tested on every odorant, so the measurement pattern is
uniform — no assay-block confound, no sparsity. We remove (b) explicitly by **residualising**:
the target is each receptor's response to the held-out class *above its own baseline
responsiveness*. A model that only knows tuning cannot predict that residual.

**Design.** For a chemically-coherent odorant class C: drop its columns, train the signed
graph on the remaining odorants (titular **q99** and full-coverage **q0**, 5 seeds), get the
refined receptor embedding — which never saw a class-C odorant. Then predict each receptor's
held-out C-response from its neighbours in the embedding (leave-one-out kNN), and score the
correlation, **raw** (tuning-inflated) and **residualised** (specificity — the honest test).
Baselines: **raw ESM** (structure only, sees no binding) and a **row-shuffle permutation
null**. If refined beats ESM on the *residualised* score, the graph transferred a specific
mechanism to an unseen ligand class. Small N (esp. HC=24) and strong receptor-correlation
(esp. CC) mean we read continuous correlations with per-seed CIs, not binary AUROC.


In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
from scipy.stats import t as _t

from orbind.regimes_ofm import ofm_pairs
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict, _train_one, MOL, PROT)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# ---- configuration -------------------------------------------------------
DATASETS = ["cc", "hc"]                 # Carey (mosquito), Hallem-Carlson (fly)
VARIANTS = {
    "q99 (titular)": dict(q=0.99, criterion="greedy_pair_cover", k_mode="coverage_quantile"),
    "full (q=0)":    dict(q=0.0,  criterion="coverage",          k_mode="coverage_quantile"),
}
SEEDS   = [42, 43, 44, 45, 46]
EPOCHS  = 900
K_NN    = 5
N_PERM  = 100            # row-shuffle permutations for the null band

SMARTS = {
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "alcohol":         "[#6][OX2H]",
    "aldehyde":        "[CX3H1](=O)[#6]",
    "ketone":          "[#6][CX3](=O)[#6]",
    "ester":           "[#6][CX3](=O)[OX2H0][#6]",
    "aromatic":        "c1ccccc1",
}
MIN_MEMBERS = 6         # a class must have >= this many odorants to be held out
FOCUS   = ["carboxylic_acid"]   # only the flagship is trained; other classes' numbers are
                                # already recorded, so we do not re-train on them

## 1 · Helpers: data, chemical classes, graph training, evaluation


In [ ]:
def load_ds(ds):
    p = ofm_pairs(ds)
    recs = sorted(p["receptor"].unique())
    ods  = sorted(p["inchikey"].unique())
    r_i  = {r: i for i, r in enumerate(recs)}
    o_i  = {o: j for j, o in enumerate(ods)}
    g = p.groupby(["receptor", "inchikey"], as_index=False)["label"].mean()
    R = np.full((len(recs), len(ods)), np.nan, np.float32)
    R[g["receptor"].map(r_i), g["inchikey"].map(o_i)] = g["label"].to_numpy()
    smi = p.drop_duplicates("inchikey").set_index("inchikey")["smiles"].to_dict()
    return p, recs, ods, R, smi

def classes_for(ods, smi):
    out = {}
    for cname, sm in SMARTS.items():
        q = Chem.MolFromSmarts(sm)
        members = [o for o in ods
                   if (m := Chem.MolFromSmiles(smi[o])) is not None and m.HasSubstructMatch(q)]
        if len(members) >= MIN_MEMBERS:
            out[cname] = set(members)
    return out

def train_refined(ds, p, variant_kw, seed, drop_iks):
    """Signed graph trained on all pairs whose ODORANT is not in drop_iks (held-out class
    removed); returns {receptor: refined vector}, {receptor: raw ESM vector}."""
    ext = GnnSignedExtractor(name="cls",
        protein_path=str(DATA / f"embeddings/proteins/esm1b_650m_mean_{ds}.npz"),
        molecule_path=str(DATA / f"embeddings/molecules/gin_supervised_contextpred_{ds}.npz"),
        emit="prot", n_models=1, hidden=256, edge_threshold=0.0, task="regression",
        epochs=EPOCHS, **variant_kw)
    idx = np.where((~p["inchikey"].isin(drop_iks)).to_numpy())[0]
    idx = idx[ext.covered(p, idx)]
    m2i, p2i, xm, xp = _build_universe(p, idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(p, idx, m2i, p2i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pe, ne = _edge_index_dict(pos, neg)
    sub = p.iloc[idx]
    torch.manual_seed(seed)
    _, zp, _ = _train_one(ext._build_model, xm, xp, pe, ne,
                          sub["inchikey"].map(m2i).to_numpy(),
                          sub["receptor"].map(p2i).to_numpy(),
                          sub["label"].to_numpy(np.float32), ext._hp(seed), DEVICE, None)
    return {r: zp[i] for r, i in p2i.items()}, {r: xp[i].numpy() for r, i in p2i.items()}

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def knn_pred(X, target, k):
    S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    return np.stack([target[nn[i]].mean(0) for i in range(len(target))])

def col_corr(actual, pred):
    cs = []
    for o in range(actual.shape[1]):
        a, q = actual[:, o], pred[:, o]
        if a.std() > 1e-8 and q.std() > 1e-8:
            cs.append(np.corrcoef(a, q)[0, 1])
    return float(np.mean(cs)) if cs else np.nan

def perm_null(X, target, k, n=N_PERM):
    base = col_corr(target, knn_pred(X, target, k))
    rng = np.random.default_rng(0)
    null = []
    for _ in range(n):
        Xs = X[rng.permutation(len(X))]
        null.append(col_corr(target, knn_pred(Xs, target, k)))
    return base, float(np.mean(null)), float(np.std(null))

## 1b · See the class: response table + molecular skeletons

Before modelling, look at the carboxylic acids directly. The **complete** insect matrix makes
this crisp: reorder the odorants so the acids form a block on the left and sort receptors by
their acid response — if a distinct band of receptors lights up on the acid block but not
elsewhere, the class is functionally atypical (this is what `funcRedund` measured, now by
eye). Then the 2D structures: acids all carry the same `-C(=O)-O-H` hook, versus the nearest
non-acids and random odorants.


In [ ]:
from rdkit.Chem import Draw, AllChem, DataStructs
CVIZ = "carboxylic_acid"

for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cl = classes_for(ods, smi)
    if CVIZ not in cl:
        print(ds, "no", CVIZ); continue
    o_i    = {o: j for j, o in enumerate(ods)}
    ac     = [o_i[o] for o in ods if o in cl[CVIZ]]
    rest   = [j for j in range(len(ods)) if j not in ac]
    rorder = np.argsort(-np.nan_to_num(np.nanmean(R[:, ac], axis=1)))   # receptors by acid response
    Mv     = R[np.ix_(rorder, ac + rest)]
    fig, ax = plt.subplots(figsize=(13, 4.8))
    im = ax.imshow(Mv, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
    ax.axvline(len(ac) - 0.5, color="black", lw=2.5)
    ax.set_title(f"{ds.upper()} responses (z-score); receptors sorted by acid response — acid block left of the line")
    ax.set_xlabel(f"odorants   [ {len(ac)} carboxylic acids | {len(rest)} others ]")
    ax.set_ylabel(f"{len(recs)} receptors")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, fraction=0.02, label="response (z)"); fig.tight_layout(); plt.show()

    acids  = [o for o in ods if o in cl[CVIZ]]
    others = [o for o in ods if o not in cl[CVIZ]]
    def fp(ik):
        m = Chem.MolFromSmiles(smi[ik]); return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m else None
    def show_grid(iks, title, n=18):
        ms = [Chem.MolFromSmiles(smi[ik]) for ik in iks[:n]]; ms = [m for m in ms if m is not None]
        print(f"[{ds.upper()}] {title}"); display(Draw.MolsToGridImage(ms, molsPerRow=6, subImgSize=(150, 110)))
    fa   = [fp(o) for o in acids if fp(o) is not None]
    near = sorted(((o, max(DataStructs.BulkTanimotoSimilarity(fp(o), fa))) for o in others if fp(o) is not None),
                  key=lambda t: -t[1])
    rng = np.random.default_rng(0)
    show_grid(acids, f"carboxylic acids — held-out class ({len(acids)}); same -COOH hook")
    show_grid([o for o, _ in near], "nearest NON-acid odorants (top Tanimoto to acids)")
    show_grid(list(rng.choice(others, min(18, len(others)), replace=False)), "random odorants")

## 1c · The picture: actual acid responses vs our prediction vs ESM (full GNN)

Train the **full-coverage (q=0)** graph with all acids removed, then reconstruct each
receptor's acid response block from its neighbours in the embedding (leave-one-out kNN).
Three panels, same z-score palette, receptors sorted by acid response: **actual** | **ours**
(full GNN, acids never seen) | **ESM**. Our panel should track the actual band; ESM's should
be flat/wrong.


In [ ]:
for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cl = classes_for(ods, smi)
    if CVIZ not in cl:
        continue
    ref_v, raw_v = train_refined(ds, p, VARIANTS["full (q=0)"], 42, cl[CVIZ])
    order = [r for r in recs if r in ref_v]; ri = [recs.index(r) for r in order]
    o_i = {o: j for j, o in enumerate(ods)}; ac = [o_i[o] for o in ods if o in cl[CVIZ]]
    A  = R[np.ix_(ri, ac)]                                       # actual acid responses
    Xr = np.stack([ref_v[r] for r in order]); Xe = np.stack([raw_v[r] for r in order])
    def reconstruct(X):
        S = emb_sim(X).copy(); np.fill_diagonal(S, -np.inf)
        nn = np.argpartition(-S, K_NN, axis=1)[:, :K_NN]
        return np.stack([A[nn[i]].mean(0) for i in range(len(order))])
    P_our, P_esm = reconstruct(Xr), reconstruct(Xe)
    ro = np.argsort(-np.nan_to_num(A.mean(1)))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, (Mx, ttl) in zip(axes, [(A, "actual"), (P_our, "OUR (full GNN, acids unseen)"), (P_esm, "ESM")]):
        ax.imshow(Mx[ro], aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
        ax.set_title(f"{ds.upper()} - {ttl}", fontsize=10); ax.set_xlabel(f"{len(ac)} acids")
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(f"{len(order)} receptors (sorted by acid response)")
    fig.tight_layout(); plt.show()

## 2 · Run the holdout: per dataset x class x variant x seed

For each held-out class we build the held-out response block `Ymat` (receptors x class
odorants) and its residual `resid = Ymat - baseline`, where baseline is the receptor's mean
response over the *retained* odorants (its general responsiveness). We score each embedding
on both.


In [ ]:
rows = []
null_rows = []
for ds in DATASETS:
    p, recs, ods, R, smi = load_ds(ds)
    cls = {c: v for c, v in classes_for(ods, smi).items() if c in FOCUS}
    print(f"\n=== {ds.upper()} ===  {len(recs)} receptors x {len(ods)} odorants | "
          f"classes: {{{', '.join(f'{c}:{len(v)}' for c, v in cls.items())}}}")
    o_i = {o: j for j, o in enumerate(ods)}
    for cname, iks in cls.items():
        cols  = [o_i[o] for o in ods if o in iks]
        keep  = [j for j in range(len(ods)) if j not in cols]

        def targets_for(order):
            ri    = [recs.index(r) for r in order]
            Ymat  = np.nan_to_num(R[np.ix_(ri, cols)])
            base  = np.nan_to_num(np.nanmean(R[np.ix_(ri, keep)], axis=1, keepdims=True))
            return Ymat, Ymat - base

        first = True
        for vname, vkw in VARIANTS.items():
            for seed in SEEDS:
                ref, raw = train_refined(ds, p, vkw, seed, iks)
                order = [r for r in recs if r in ref]
                Ymat, resid = targets_for(order)
                Xr = np.stack([ref[r] for r in order])
                Xe = np.stack([raw[r] for r in order])
                if first:
                    rows.append(dict(ds=ds, cls=cname, emb="raw ESM", variant="-", seed="-",
                                     raw=col_corr(Ymat, knn_pred(Xe, Ymat, K_NN)),
                                     spec=col_corr(resid, knn_pred(Xe, resid, K_NN))))
                    _, nm, ns = perm_null(Xe, resid, K_NN)
                    null_rows.append(dict(ds=ds, cls=cname, null_mean=nm, null_sd=ns))
                    first = False
                rows.append(dict(ds=ds, cls=cname, emb=f"refined {vname}", variant=vname, seed=seed,
                                 raw=col_corr(Ymat, knn_pred(Xr, Ymat, K_NN)),
                                 spec=col_corr(resid, knn_pred(Xr, resid, K_NN))))
        for vname in VARIANTS:
            sub = [r["spec"] for r in rows if r["ds"] == ds and r["cls"] == cname and r["variant"] == vname]
            esm = [r["spec"] for r in rows if r["ds"] == ds and r["cls"] == cname and r["emb"] == "raw ESM"][0]
            print(f"  {cname:16} {vname:14} spec r={np.nanmean(sub):+.3f} (ESM {esm:+.3f})")
res  = pd.DataFrame(rows)
null = pd.DataFrame(null_rows)

## 3 · Results — specificity (residualised) vs raw

`raw` = predict held-out response with tuning left in (inflated; refined wins mostly because it
knows tuning breadth). `spec` = predict the residual (specific mechanism, tuning removed) — the
honest test. Bars show the residualised score; the dotted line is the permutation null.


In [ ]:
def ci(x):
    x = np.asarray([v for v in x if np.isfinite(v)], float)
    if len(x) < 2: return (np.nanmean(x) if len(x) else np.nan, np.nan)
    return x.mean(), _t.ppf(.975, len(x) - 1) * x.std(ddof=1) / len(x) ** .5

# summary table (both raw and spec)
sm = []
for (ds, cname), _ in res.groupby(["ds", "cls"]):
    row = {"ds": ds, "class": cname}
    for metric in ["raw", "spec"]:
        row[f"ESM_{metric}"] = res[(res.ds==ds)&(res.cls==cname)&(res.emb=="raw ESM")][metric].iloc[0]
        for vname in VARIANTS:
            m, _ = ci(res[(res.ds==ds)&(res.cls==cname)&(res.variant==vname)][metric])
            row[f"{vname}_{metric}"] = m
    sm.append(row)
sm = pd.DataFrame(sm)
display(sm.style.format({c: "{:+.3f}" for c in sm.columns if c not in ("ds", "class")}).hide(axis="index"))

for ds in DATASETS:
    classes = sm[sm.ds == ds]["class"].tolist()
    if not classes: continue
    fig, ax = plt.subplots(figsize=(1.6*len(classes)+4, 4.3))
    labels = ["raw ESM"] + [f"refined {v}" for v in VARIANTS]
    colors = {"raw ESM": "#888888", labels[1]: "#4c78a8", labels[2]: "#e4794c"}
    x = np.arange(len(classes)); w = 0.26
    for j, lab in enumerate(labels):
        vals, errs = [], []
        for c in classes:
            if lab == "raw ESM":
                vals.append(res[(res.ds==ds)&(res.cls==c)&(res.emb=="raw ESM")]["spec"].iloc[0]); errs.append(0)
            else:
                vname = lab.replace("refined ", "")
                m, h = ci(res[(res.ds==ds)&(res.cls==c)&(res.variant==vname)]["spec"]); vals.append(m); errs.append(h)
        ax.bar(x + (j-1)*w, vals, w, yerr=errs, capsize=3, label=lab, color=colors[lab])
    for xi, c in enumerate(classes):
        nm = null[(null.ds==ds)&(null.cls==c)]["null_mean"].iloc[0]
        ax.plot([xi-0.45, xi+0.45], [nm, nm], ls=":", c="gray", lw=1)
    ax.axhline(0, c="k", lw=.6)
    ax.set_xticks(x); ax.set_xticklabels(classes)
    ax.set_ylabel(f"specificity: held-out C-response corr (LOO {K_NN}-NN, tuning removed)")
    ax.set_title(f"{ds.upper()} — mechanism transfer to an UNSEEN odorant class (dotted = null)")
    ax.legend(frameon=False); fig.tight_layout(); plt.show()

**Reading it.** On the **raw** metric refined beats ESM easily — but that is largely tuning
breadth, which we already knew the graph encodes; it is *not* the mechanism claim. The
**specificity** bars are the real test: predicting a receptor's held-out class response *above
its own baseline*, a signal tuning cannot supply. If refined clears both the raw-ESM bar **and**
the permutation null there, the receptor embedding carries a specific, transferable binding
mechanism — demonstrated on a complete matrix (no measurement confound) with tuning explicitly
removed. Where refined only matches ESM/null on specificity, the honest conclusion is that the
transferable signal is generic responsiveness, not a class-specific mechanism. Comparing q99 vs
q0 shows whether the hub-molecule restriction helps this transfer. Read CC with care: its
receptors are strongly cross-correlated, which inflates absolute correlations and shrinks the
effective N — the ESM baseline and the null are what keep the comparison fair.
